# Outlier Analysis (topic = −1) — Combined

Documents assigned to topic **−1** by HDBSCAN are *noise* — they did not fall
inside any dense cluster. This notebook profiles those outliers across **both**
corpora (iGEM teams and SynBio papers) to understand **why** the outlier rate is
high and whether any of them could reasonably be reassigned.

Run the teams and papers topic-model notebooks first so the
`assets/topic_models/*_doc_topics.txt` files exist.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

## 1. Load data

In [ ]:
import pandas as pd
from IPython.display import display

from aux.paths import SEED, set_seed
from aux.orphans import (
    load_outlier_inputs, outlier_summary, plot_outlier_rate_by_year,
    plot_text_length, language_outlier_table, nearest_centroid_analysis,
    citation_profile, concept_overlap, sample_outlier_docs,
    sample_outlier_table, save_orphans, build_summary,
)

set_seed()

papers, papers_emb, papers_topics = load_outlier_inputs(
    raw_file="synbio_openalex.txt",
    embeddings_file="papers_embeddings.npy",
    doc_topics_file="papers_doc_topics.txt",
    id_col="id",
)
teams, teams_emb, teams_topics = load_outlier_inputs(
    raw_file="igem.txt",
    embeddings_file="teams_embeddings.npy",
    doc_topics_file="teams_doc_topics.txt",
    id_col="UT",
)
print(f"Papers: {len(papers):,}  |  Teams: {len(teams):,}")

## 2. Outlier overview

In [ ]:
outlier_summary(papers, "Papers", "publication_year")
outlier_summary(teams, "Teams", "PY")

## 3. Outlier rate by year

If outliers concentrate in certain years the data may simply be sparse or the
vocabulary may differ in those periods.

In [ ]:
plot_outlier_rate_by_year(papers, "publication_year", "Papers — outlier rate by year")
plot_outlier_rate_by_year(teams, "PY", "Teams — outlier rate by year")

## 4. Text length: outliers vs assigned

Short or empty abstracts often end up as outliers because their embeddings carry
little signal.

In [ ]:
plot_text_length(papers, "abstract", "Papers — abstract word count")
print()
plot_text_length(teams, "AB", "Teams — abstract word count")

## 5. Language distribution (papers only)

Non-English papers encoded with an English sentence-transformer may produce
out-of-distribution embeddings that cluster poorly.

In [ ]:
lang = language_outlier_table(papers)
if lang is not None:
    display(lang)

## 6. Embedding distance to nearest cluster centroid

For each outlier document we measure the cosine distance to the nearest cluster
centroid. Documents close to a centroid might be recoverable; truly isolated
documents are genuine noise.

In [ ]:
print("── Papers ─────────────────────────────────────────")
papers_dists_out, papers_dists_asg = nearest_centroid_analysis(
    papers_emb, papers_topics["topic"].values, "Papers"
)
print()
print("── Teams ──────────────────────────────────────────")
teams_dists_out, teams_dists_asg = nearest_centroid_analysis(
    teams_emb, teams_topics["topic"].values, "Teams"
)

## 7. Citation profile of outliers (papers)

Low-citation outliers may be marginal or off-topic. High-citation outliers are
worth investigating further.

In [ ]:
display(citation_profile(papers))

## 8. Concept overlap (papers)

OpenAlex assigns concept tags to each paper. If outlier papers carry very
different concepts they may be genuinely off-topic.

In [ ]:
display(concept_overlap(papers))

## 9. Sample outlier documents

A random sample of outlier titles/abstracts helps build intuition about what
kind of documents are being left out.

In [ ]:
sample_outlier_docs(papers, "title", "abstract", "publication_year", "PAPERS", k=10, seed=SEED)
sample_outlier_docs(teams, "TI", "AB", "PY", "TEAMS", k=10, seed=SEED)

In [ ]:
# Random sample of outlier teams (table view)
sample = sample_outlier_table(
    teams, ["UT", "PY", "TI", "AB"], k=20, seed=SEED,
    truncate_col="AB", truncate_len=300,
)
if sample is None:
    print("No outlier teams to display.")
else:
    display(sample)

## Save orphan documents

In [ ]:
orphan_papers = save_orphans(
    papers, ["id", "publication_year", "title", "abstract"], "orphans_papers.tsv"
)
orphan_teams = save_orphans(teams, ["UT", "PY", "TI", "AB"], "orphans_teams.tsv")
print(f"Saved {len(orphan_papers):,} orphan papers → orphans_papers.tsv")
print(f"Saved {len(orphan_teams):,} orphan teams  → orphans_teams.tsv")

## 10. Summary

Collect the key findings into a single table.

In [ ]:
summary = pd.DataFrame([
    build_summary(papers, papers_emb, papers_topics["topic"].values, "Papers", "abstract"),
    build_summary(teams, teams_emb, teams_topics["topic"].values, "Teams", "AB"),
]).set_index("dataset")
display(summary)